# Deep Agents: Building a Research Agent (Workshop Edition)

<img src="./images/deepAgentsDiag.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents harness overview">

In the next ~30 minutes, you'll progressively build a research agent with [LangChain Deep Agents](https://docs.langchain.com/oss/python/deepagents/).

**What you'll learn:** the harness, custom tools, backends, subagents, human-in-the-loop, long-term memory, and AGENTS.md + Skills.

> Requires a [Tavily API key](https://tavily.com) for web search and an OpenAI key for the default model (`gpt-5.4`).


## Part 0: Setup

If you haven't already: install dependencies and copy `.env.example` → `.env`. See the README.

The model is configured in `utils/models.py` — defaults to `openai:gpt-5.4` for the main agent. The research subagent in Part 4 uses `openai:gpt-5.4-mini`. Edit `utils/models.py` to swap providers.


In [ ]:
# Quick install if needed:
# !pip install -e .

import sys
from pathlib import Path
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from utils.models import model
print(model)

from dotenv import load_dotenv
load_dotenv(override=True)

## Part 1: Your First Deep Agent (The Harness)

<img src="./images/deepAgentsHarnessOverview.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Deep Agents harness overview">

Deep Agents is an **agent harness** — a tool-calling loop with pre-built tools and capabilities baked in.

**You get out of the box:**
- **Filesystem tools** — `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`
- **Planning tool** — `write_todos` for task tracking
- **Subagent delegation** — `task()` tool for isolated work
- **Context management** — auto-evicts large tool results (>20k tokens) to the filesystem and summarizes history at ~85% context capacity

These come from a stack of built-in middleware (TodoList, Filesystem, SubAgent, Summarization, PatchToolCalls) automatically attached when you call `create_deep_agent()`. You can also pass your own via `middleware=[...]`.



In [ ]:
from deepagents import create_deep_agent

agent = create_deep_agent(
    model=model
)

# Even with no custom tools, the agent can write/read files
result = agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called notes.md with 'Hello from Deep Agents!' then read it back."}]
})

print("Response:", result["messages"][-1].content)
print("\nVirtual filesystem:", list(result.get("files", {}).keys()))


**Key takeaway:** `create_deep_agent()` gives you filesystem + planning capabilities, and files default to ephemeral agent state — we'll customize that in Part 3.

## Part 2: Adding Custom Tools

A research agent needs **web search**. Let's add a Tavily tool with the `@tool` decorator.


In [ ]:
from langchain.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool(parse_docstring=True)
def tavily_search(query: str) -> str:
    """Search the web for information on a given query.

    Args:
        query: Search query to execute
    """
    results = tavily_client.search(query, max_results=3, topic="general")
    chunks = [
        f"## {r['title']}\n**URL:** {r['url']}\n\n{r.get('content', '')}\n\n---\n"
        for r in results.get("results", [])
    ]
    return f"Found {len(chunks)} result(s) for '{query}':\n\n{''.join(chunks)}"

agent = create_deep_agent(
    model=model,
    tools=[tavily_search]
)

# Quick test - keep prompts succinct to keep demos fast
result = agent.invoke({
    "messages": [{"role": "user", "content": "In 2 sentences, what is LangGraph? Use one search."}]
})
print(result["messages"][-1].content)


**Key takeaway:** the `@tool` decorator turns any Python function into a LangChain tool — pass tools via `tools=[...]`.


## Part 3: Understanding Backends

<img src="./images/deepAgentBackends.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Backend Architecture">

Where do the agent's files actually go? **Backends** are pluggable storage. The question that actually matters when choosing one: **who can see the files?**

| Backend | Who can see them | Use case |
|---|---|---|
| **StateBackend** *(default)* | This thread only | Scratch pad, intermediate results, large tool-output eviction |
| **FilesystemBackend** | Anything with access to that directory | Local projects, CI sandboxes, mounted volumes |
| **StoreBackend** | Any thread in the same namespace | Long-term memories (Part 6) |
| **CompositeBackend** | Depends on the route | Mix backends (Part 6) |


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langsmith import uuid7

# Checkpointer lets us persist within a thread; thread_id selects the conversation
checkpointer = MemorySaver()

agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a helpful research assistant.",
    checkpointer=checkpointer,
)

# Thread 1: write a file
config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Write a file called /research_notes.md with 'My research findings go here'"}]
}, config=config1)
print("Thread 1 files:", list(result.get("files", {}).keys()))

# Thread 2: brand-new thread → file is gone (StateBackend is ephemeral!)
config2 = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "List all files with ls /"}]
}, config=config2)
print("\nThread 2 (new thread):", result["messages"][-1].content)


**Key takeaway**: the choice between backends is about who can see the files - just this thread (StateBackend), anyone with disk access (FilesystemBackend), or other threads in the same namespace (StoreBackend). CompositeBackend mixes them.

### FilesystemBackend — writing to real disk

When the agent needs to work with **actual files**, use `FilesystemBackend` with `virtual_mode=True` to sandbox under `root_dir`.

> ⚠️ Always sandbox with `virtual_mode=True` to prevent path traversal.


In [ ]:
from deepagents.backends import FilesystemBackend
import tempfile, os, shutil

sandbox_dir = tempfile.mkdtemp(prefix="deepagents_sandbox_")
fs_backend = FilesystemBackend(root_dir=sandbox_dir, virtual_mode=True)

agent_with_fs = create_deep_agent(
    model=model,
    backend=fs_backend,
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": str(uuid7())}}
agent_with_fs.invoke({
    "messages": [{"role": "user", "content": "Write notes.txt with 'Hello from FilesystemBackend!'"}]
}, config=config)

# Verify it actually hit disk
actual_path = os.path.join(sandbox_dir, "notes.txt")
with open(actual_path) as f:
    print(f"File on disk at {actual_path}: {f.read()}")

In [ ]:
#Remove the file 
shutil.rmtree(sandbox_dir, ignore_errors=True)


## Part 4: Adding a Research Subagent

<img src="./images/deepAgentSubagents.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Subagent Architecture">

As agents do more work, their context fills with intermediate tool calls. **Subagents** isolate work in a separate context — only the final summary returns to the main agent.

```
Without subagents: every search result balloons the main agent's context
With subagents:    subagent searches in isolation, returns one clean summary
```

**Key takeaway:** delegate via `subagents=[...]`. The main agent calls the subagent through a `task()` tool and only sees the result, not the intermediate tool calls.


In [ ]:
from utils.models import sub_agent_model
print(sub_agent_model)

In [ ]:
research_subagent = {
    "name": "research-agent",
    "description": "Delegate focused research tasks.",
    "system_prompt": (
        f"You are a research assistant. \n"
        "- Use 1-2 searches maximum, then summarize.\n"
        "- Cite with inline numbers [1], [2] and end with a Sources section."
    ),
    "tools": [tavily_search],
    "model": sub_agent_model,
}

agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt="You are a research coordinator. Always delegate research to subagents.",
    subagents=[research_subagent],
    checkpointer=checkpointer,
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent.invoke({
    "messages": [{"role": "user", "content": "Research what LangGraph and LangChain are and give me a 1-paragraph summary for each."}]
}, config=config)
print(result["messages"][-1].content)


## Part 5: Human-in-the-Loop

<img src="./images/deepAgentHITL.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Human-in-the-Loop">

For sensitive operations, you may want a human to approve before the agent acts. Deep Agents has interrupt support built in via `HumanInTheLoopMiddleware`.

**Built-in decision types:** Approve · Edit · Reject. You can scope per tool:

```python
interrupt_on = {
    "delete_file": {"allowed_decisions": ["approve", "edit", "reject"]},
    "write_file": {"allowed_decisions": ["approve", "reject"]},
}
```

**Key takeaway:** configure `interrupt_on={...}` to gate risky tools. A checkpointer is required (the agent must pause and resume).


In [ ]:
# Interrupt on file writes/edits
agent_with_hitl = create_deep_agent(
    model=model,
    tools=[tavily_search],
    checkpointer=checkpointer,
    interrupt_on={"write_file": True, "edit_file": True},
)

config = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_hitl.invoke({
    "messages": [{"role": "user", "content": "Write a file called /test.md with 'Hello World'"}]
}, config=config)

if result.get("__interrupt__"):
    print("Interrupt triggered!")
    iv = result["__interrupt__"][0].value
    for action, review in zip(iv["action_requests"], iv["review_configs"]):
        print(f"  Tool: {action['name']}  Args: {action['args']}")
        print(f"  Allowed: {review['allowed_decisions']}")


In [ ]:
from langgraph.types import Command

if result.get("__interrupt__"):
    result = agent_with_hitl.invoke(
        Command(resume={"decisions": [{"type": "approve"}]}),
        config=config,
    )
    print("Resumed:", result["messages"][-1].content)

files = result.get("files", {})
print("Files now in state:", list(files.keys()))
file_data = files.get("/test.md") or {}
print("File content:", file_data.get("content", "(not found)"))


## Part 6: Long-Term Memory

<img src="./images/deepAgentMemories.png" style="width: auto; height: 420px; border-radius: 8px;" alt="Long-Term Memory">

Files in StateBackend disappear when a thread ends. **Long-term memory** uses `CompositeBackend` to route specific paths to a persistent `StoreBackend`.

```
/memories/*       →  StoreBackend  (persistent, cross-thread)
everything else   →  StateBackend  (ephemeral)
```

**Key takeaway:** mix backends with `CompositeBackend` and pass a `store=` for the persistent layer. In production, LangGraph Platform / LangSmith Deployments provides the store automatically.


In [ ]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StateBackend, StoreBackend, CompositeBackend

store = InMemoryStore()

# Compose: /memories/* persists in the store; everything else stays ephemeral
backend = CompositeBackend(
    default=StateBackend(),
    routes={"/memories/": StoreBackend()},
)

agent_with_memory = create_deep_agent(
    model=model,
    tools=[tavily_search],
    system_prompt=(
        "You are a helpful assistant with long-term memory.\n"
        "Save important notes to /memories/ so they persist across conversations.\n"
        "Files outside /memories/ disappear when the conversation ends."
    ),
    checkpointer=checkpointer,
    backend=backend,
    store=store,
)

# Thread 1: save to memory
config1 = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_memory.invoke({
    "messages": [{"role": "user", "content": "Save 'AI agents are evolving rapidly in 2026' to `/memories/findings.md`"}]
}, config=config1)
print("Thread 1:", result["messages"][-1].content)

# Prove the routing: StateBackend stays empty; the store holds the file.
state_files = list(result.get("files", {}).keys())
print(f"\nStateBackend files (ephemeral, per-thread): {state_files}")

store_items = store.search(("filesystem",))
print(f"StoreBackend items (persistent, cross-thread):")
for item in store_items:
    content = item.value.get("content", "")
    print(f"  {item.key} -> {content!r}")


In [ ]:
# Thread 2: brand-new thread, but the memory persists!
config2 = {"configurable": {"thread_id": str(uuid7())}}
result = agent_with_memory.invoke({
    "messages": [{"role": "user", "content": "Read me what's in `/memories/findings.md`"}]
}, config=config2)
print("Thread 2:", result["messages"][-1].content)


## Part 7: AGENTS.md & Skills

So far we've put instructions in `system_prompt`. Two file-based alternatives are more powerful:

| Approach | Loaded when | Editable by agent | Best for |
|---|---|---|---|
| `system_prompt` | Always | No | Core identity, immutable rules |
| `AGENTS.md` (`memory=`) | Always | **Yes** | Workflow, learnable rules |
| `SKILL.md` (`skills=`) | On demand | No | Task-specific templates |

- **AGENTS.md** is loaded into the system prompt via `memory=`; the agent can read AND edit its own AGENTS.md.
- **Skills** use **progressive disclosure**: only the skill's name + description loads at startup; the full SKILL.md is read on demand when the agent decides it's relevant.



In [ ]:
# AGENTS.md — the agent's identity + workflow (always loaded)
agents_md = """# Research Assistant

You search the web, synthesize findings, and produce polished content.

## Workflow
1. **Plan** — use `write_todos` to break down the task
2. **Research** — search via tavily_search (1-2 searches max)
3. **Synthesize** — combine findings into a brief report
4. **Remember** — save key takeaways to `/memories/research_notes.md`

## Rules
- Cite with inline numbers [1], [2]; consolidate duplicate URLs
- End reports with a Sources section
- Check loaded skills before formatting specialized content
"""

# A skill — loaded on demand when the agent sees a LinkedIn-shaped task
linkedin_skill = """---
name: linkedin-post
description: Write a LinkedIn post from research findings. Use for professional posts or thought-leadership.
---

# LinkedIn Post Skill

## Format
- Hook line that grabs attention before the 'see more' cut
- 3-5 short paragraphs (1-2 sentences each), with line breaks between
- 1-2 emojis per paragraph (don't overdo)
- End with a question or CTA + 3-5 hashtags

## Length
- 150-300 words. First line must hook the reader.
"""

In [ ]:
from deepagents.backends.utils import create_file_data

skill_agent = create_deep_agent(
    model=model,
    tools=[tavily_search],
    memory=["/AGENTS.md"],
    skills=["/skills/"],
    checkpointer=checkpointer,
    backend=backend,
    store=store,
)

# Seed the virtual filesystem so memory= and skills= can find the files
files = {
    "/AGENTS.md": create_file_data(agents_md),
    "/skills/linkedin-post/SKILL.md": create_file_data(linkedin_skill),
}

config = {"configurable": {"thread_id": str(uuid7())}}
result = skill_agent.invoke({
    "messages": [{"role": "user", "content": "Briefly research what LangChain is, then write a short LinkedIn post about them."}],
    "files": files,
}, config=config)
print(result["messages"][-1].content)

## Part 8: Wrap-Up & Next Steps

Starting from a basic `create_deep_agent()`, we progressively added:

```
Part 1: create_deep_agent(model)              → Filesystem + planning
Part 2: + tools=[tavily_search]               → Web search
Part 3: (backends)                            → Storage model: state vs disk
Part 4: + subagents=[research_subagent]       → Context isolation
Part 5: + interrupt_on={...}                  → Human oversight
Part 6: + backend=CompositeBackend + store    → Long-term memory
Part 7: + memory=AGENTS.md + skills=SKILL.md  → File-based identity + on-demand capabilities
```

### Resources
- [Deep Agents docs](https://docs.langchain.com/oss/python/deepagents/)
- [LangChain Academy](https://academy.langchain.com/)
- [LangChain vs LangGraph vs Deep Agents](https://docs.langchain.com/oss/python/concepts/products)

### What to try next
1. **Run in Studio** — wrap your agent for `langgraph dev` or LangSmith Deployments
2. **More skills** — write SKILL.md files for your domain
3. **Per-user memory** — namespace `StoreBackend` by `user_id`
4. **Multi-agent systems** — compose multiple specialized subagents (try using a cheaper open model for subagents!)

---

**Happy building!** 🚀
